In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import shutil
import cv2

# Add the src directory to the path. TEMPORARY FIX
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

from src.data_processing.dataset_loader import CoastData

In [ ]:
data_path = os.path.abspath(os.path.join(os.getcwd(), "../../../data/SCLabels_oblique_v1.0.0/"))

registered_path =  os.path.abspath(os.path.join(os.getcwd(), "../../../data/registered_images/"))
sclabels_path = os.path.abspath(os.path.join(os.getcwd(), "../../../data/SCLabels_oblique_v1.0.0/"))

# # Test with a single station. Differents stations: agrelo, cies, cadiz, samarador, arenaldentem
destination_path = os.path.abspath(os.path.join(os.getcwd(), "../../../data/SCLabels_oblique_registered_v1.0.0/images/"))

stations = ["agrelo", "arenaldentem", "cadiz", "cies", "samarador"]

for station in stations:
    print(f"Processing station: {station}")
    data = CoastData(sclabels_path, name=station)
    filtered_data = data.get_images(original_image_path=True, filename=True)
    original_file_names = []
    file_names = []
    for item in filtered_data:
        original_file_names.append(item['original_image_path'])
        file_names.append(item['filename'])

    print(f"{len(filtered_data)} images found.")
    print(f"{len(file_names)} images found.")

    path = os.path.join(registered_path, f"Registered_{station}", "Registered")

    # get list of folders
    folders = sorted(os.listdir(path))

    for folder in folders:
        print(f"Processing folder: {folder}")
        folder_path = os.path.join(path, folder)

        # get images in folder
        images = sorted(os.listdir(folder_path))

        for i, image_name in enumerate(images):
            if image_name in original_file_names:
                index_in_files = original_file_names.index(image_name)
                image_path = os.path.join(folder_path, image_name)
                aux_destination_path = os.path.join(destination_path, file_names[index_in_files])
                # copy image to another folder
                shutil.copy(image_path, aux_destination_path)
            else:
                print(f"Image {image_name} not found in files_name.")

In [ ]:
new_path = os.path.abspath(os.path.join(os.getcwd(), "../../../data/SCLabels_oblique_registered_v1.0.0/"))

data = CoastData(new_path, name="arenaldentem")
data = data.get_images(get_mask=False, get_shoreline_coords=True, get_all_metadata=True)

# print(data)

In [ ]:
for i in range(10):
    print(f"Image: {data[i]['image']}")
    img = cv2.imread(data[i]['image'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    coords = data[i]['shoreline_coords']
    height, width, _ = img.shape

    points = []
    for u, v in zip(coords["u"], coords["v"]):
        points.append((int(u), height - int(v)))

        cv2.circle(img, (int(u), height - int(v)), 5, (255, 0, 0), 5)
    
    cv2.polylines(img, [np.array(points)], isClosed=False, color=(255, 0, 0), thickness=2)

    plt.imshow(img)
    plt.show()